## A.3. Coupling strength and DDI

Vary one delayed coupling weight while keeping the global coefficient fixed. Each condition uses one trajectory and a 20-ESN ensemble. DDI matrices are averaged before selecting the true-edge score and maximum absent-edge score.

Reuse cached observations and completed DDI results; compute only missing conditions.


In [ ]:
import hashlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from scipy.stats import spearmanr
from matplotlib.ticker import FormatStrFormatter, MaxNLocator, ScalarFormatter
%run ./time_Lorenz_data_6time.ipynb
%run ./ESN.ipynb
%run ./Reconstruct.ipynb
%run ./Plot_DDI.ipynb
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 12
N_x = 4
N_washout = 100
N_train = 10000
N_tstart = 10000
N_test = 2000
window = 6
window_length = 100
test_window = 6
N_net = 20
N_rep = 30
N_evo = 5
sigma_dyn = 1e-06
sigma_obs = 1e-06
c = 0.3
DATA_SEED = 99
OFFSET_SEED = 199
RESERVOIR_SEED = 299
DDI_PIPELINE_VERSION = 'further_study_1_full_washout_v2'
NETWORK_PATH = os.path.join('result', 'A.3', 'A.3_network.pkl')
RESULT_PATH = os.path.join('result', 'A.3', 'A.3.pkl')
FIGURE_PATH = os.path.join('result', 'figure', 'A.3.png')
os.makedirs(os.path.dirname(NETWORK_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FIGURE_PATH), exist_ok=True)


### Part 1. Load or generate the coupling conditions

Reuse `A.3_network.pkl`, or generate missing observations using paired seeds across coupling weights.


In [ ]:
if os.path.exists(NETWORK_PATH):
    with open(NETWORK_PATH, 'rb') as stream:
        network_dict = pickle.load(stream)
else:
    omega_list = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    true_t_idx = 3
    true_effect = 0
    true_cause = 1
    X_by_omega = {}
    A_by_omega = {}
    for omega in omega_list:
        np.random.seed(DATA_SEED)
        A_list = [np.zeros((N_x, N_x)) for _ in range(test_window)]
        A_list[true_t_idx][true_effect, true_cause] = float(omega)
        (A1, A2, A3, A4, A5, A6) = A_list
        X = gen_data_norm_6time(N_x, A1, A2, A3, A4, A5, A6, N_tstart + N_test, window * window_length, window_length, c=c, sigma_dyn=sigma_dyn, sigma_obs=sigma_obs)
        X_by_omega[float(omega)] = X
        A_by_omega[float(omega)] = [A.copy() for A in A_list]
    network_dict = {'omega_list': omega_list, 'A_by_omega': A_by_omega, 'effective_edge_strength': c * omega_list, 'X_by_omega': X_by_omega, 'true_t_idx': true_t_idx, 'true_delay': true_t_idx * window_length, 'true_effect': true_effect, 'true_cause': true_cause, 'config': {'N_x': N_x, 'N_washout': N_washout, 'N_train': N_train, 'N_tstart': N_tstart, 'N_test': N_test, 'window': window, 'window_length': window_length, 'test_window': test_window, 'N_net': N_net, 'N_rep': N_rep, 'N_evo': N_evo, 'sigma_dyn': sigma_dyn, 'sigma_obs': sigma_obs, 'c': c, 'DATA_SEED': DATA_SEED}}
    with open(NETWORK_PATH, 'wb') as f:
        pickle.dump(network_dict, f)


### Part 2. Compute DDI and plot

Use 20 ESNs per trajectory, 30 test histories per model and five evolution steps. Each history has a full 100-step washout. Compatible completed conditions are reused. The figure shows true-edge DDI, maximum absent-edge DDI, the fitted line and Spearman correlation. An older cache with a different history protocol is not relabelled as compatible.


In [ ]:
with open(NETWORK_PATH, 'rb') as f:
    network_data = pickle.load(f)
omega_list = np.asarray(network_data['omega_list'], dtype=float)
X_by_omega = network_data['X_by_omega']
true_t_idx = int(network_data['true_t_idx'])
true_effect = int(network_data['true_effect'])
true_cause = int(network_data['true_cause'])
n_omega = len(omega_list)
N_dim = 3 * N_x
N_dim_segments = test_window * N_dim
N_units_segments = 100 * N_dim_segments
max_lag = (test_window - 1) * window_length
test_lengths = [len(X_by_omega[float(omega)]) - max_lag - N_tstart for omega in omega_list]
max_offset = min(test_lengths) - N_evo - 1
if max_offset <= N_washout:
    raise ValueError('The embedded test sequence is too short for a full washout and rollout.')
offset_rng = np.random.RandomState(OFFSET_SEED)
offset = offset_rng.randint(N_washout, max_offset, size=(N_net, N_rep))
if np.any(offset < N_washout) or np.any(offset + N_evo >= min(test_lengths)):
    raise AssertionError('An intervention start lacks a complete washout or future interval.')
data_digest = hashlib.sha256()
for omega in omega_list:
    data_digest.update(np.asarray(X_by_omega[float(omega)]).tobytes())
calculation_config = {'ddi_pipeline_version': DDI_PIPELINE_VERSION, 'data_sha256': data_digest.hexdigest(), 'dependency_sha256': {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in ('ESN.ipynb', 'Reconstruct.ipynb')}, **{name: globals()[name] for name in ('N_x', 'N_washout', 'N_train', 'N_tstart', 'N_test', 'window', 'window_length', 'test_window', 'N_net', 'N_rep', 'N_evo', 'sigma_dyn', 'sigma_obs', 'c', 'DATA_SEED', 'OFFSET_SEED', 'RESERVOIR_SEED')}}
off_diagonal_mask = np.ones((test_window, N_x, N_x), dtype=bool)
for t_idx in range(test_window):
    np.fill_diagonal(off_diagonal_mask[t_idx], False)
true_edge_mask = np.zeros_like(off_diagonal_mask)
true_edge_mask[true_t_idx, true_effect, true_cause] = True
false_edge_mask = off_diagonal_mask & ~true_edge_mask
TDInet_by_omega = np.full((n_omega, N_net, test_window, N_x, N_x), np.nan)
if os.path.exists(RESULT_PATH):
    with open(RESULT_PATH, 'rb') as f:
        previous_result = pickle.load(f)
    previous_omega = np.asarray(previous_result.get('omega_list', []), dtype=float)
    previous_tdinet = np.asarray(previous_result.get('TDInet_by_omega', []))
    if previous_omega.shape == omega_list.shape and np.allclose(previous_omega, omega_list) and (previous_tdinet.shape == TDInet_by_omega.shape) and (previous_result.get('config') == calculation_config) and np.array_equal(previous_result.get('offset'), offset):
        TDInet_by_omega[:] = previous_tdinet
for (omega_idx, omega) in enumerate(omega_list):
    if np.all(np.isfinite(TDInet_by_omega[omega_idx])):
        continue
    X = X_by_omega[float(omega)]
    (T, D) = X.shape
    X_segments = np.zeros((T - (test_window - 1) * window_length, test_window * D))
    for i in range((test_window - 1) * window_length, T):
        segment = X[i - np.arange(test_window) * window_length].reshape(-1)
        X_segments[i - (test_window - 1) * window_length] = segment
    X_washout_segments = X_segments[:N_washout]
    X_t_segments = X_segments[N_washout:N_washout + N_train - 1]
    Y_t_segments = X_segments[N_washout + 1:N_washout + N_train]
    X_test_segments = X_segments[N_tstart:]
    TDInet = np.zeros((N_net, test_window, N_x, N_x))
    for net in range(N_net):
        np.random.seed(RESERVOIR_SEED + net)
        esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
        esn.train(X_washout_segments, X_t_segments, Y_t_segments)
        for t_idx in range(test_window):
            TDInet[net, t_idx] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 3, N_dim, N_rep, test_window, 0, 1, N_washout, N_evo, offset[net], t_idx)
    TDInet_by_omega[omega_idx] = TDInet
    completed = np.all(np.isfinite(TDInet_by_omega), axis=(1, 2, 3, 4))
    DDI_matrices = np.full((n_omega, test_window, N_x, N_x), np.nan)
    DDI_matrices[completed] = np.mean(TDInet_by_omega[completed], axis=1)
    true_edge_ddi_net = TDInet_by_omega[:, :, true_t_idx, true_effect, true_cause]
    true_edge_ddi = np.full(n_omega, np.nan)
    true_edge_ddi[completed] = DDI_matrices[completed, true_t_idx, true_effect, true_cause]
    false_edge_coordinates = np.argwhere(false_edge_mask)
    max_false_edge_ddi = np.full(n_omega, np.nan)
    max_false_edge_ddi_net = np.full((n_omega, N_net), np.nan)
    max_false_edge_index = np.full((n_omega, 3), -1, dtype=int)
    for ddi_idx in np.flatnonzero(completed):
        false_scores = DDI_matrices[ddi_idx][false_edge_mask]
        max_local_idx = int(np.argmax(false_scores))
        (t_false, effect_false, cause_false) = false_edge_coordinates[max_local_idx]
        max_false_edge_index[ddi_idx] = [t_false, effect_false, cause_false]
        max_false_edge_ddi[ddi_idx] = false_scores[max_local_idx]
        max_false_edge_ddi_net[ddi_idx] = TDInet_by_omega[ddi_idx, :, t_false, effect_false, cause_false]
    save_dict = {'omega_list': omega_list, 'DDI_matrices': DDI_matrices, 'TDInet_by_omega': TDInet_by_omega, 'true_edge_ddi': true_edge_ddi, 'true_edge_ddi_net': true_edge_ddi_net, 'max_false_edge_ddi': max_false_edge_ddi, 'max_false_edge_ddi_net': max_false_edge_ddi_net, 'max_false_edge_index': max_false_edge_index, 'true_t_idx': true_t_idx, 'true_delay': true_t_idx * window_length, 'true_effect': true_effect, 'true_cause': true_cause, 'false_edge_mask': false_edge_mask, 'offset': offset, 'config': calculation_config}
    with open(RESULT_PATH, 'wb') as f:
        pickle.dump(save_dict, f)


In [ ]:
with open(RESULT_PATH, 'rb') as f:
    loaded_data = pickle.load(f)
saved_offsets = np.asarray(loaded_data.get('offset', []))
if loaded_data.get('config', {}).get('ddi_pipeline_version') != DDI_PIPELINE_VERSION or saved_offsets.shape != (N_net, N_rep) or np.any(saved_offsets < N_washout):
    raise RuntimeError('Rerun Part 2 with the corrected full-washout offsets before plotting.')
omega_list = np.asarray(loaded_data['omega_list'], dtype=float)
true_edge_ddi = np.asarray(loaded_data['true_edge_ddi'], dtype=float)
true_edge_ddi_net = np.asarray(loaded_data['true_edge_ddi_net'], dtype=float)
max_false_edge_ddi = np.asarray(loaded_data['max_false_edge_ddi'], dtype=float)
max_false_edge_ddi_net = np.asarray(loaded_data['max_false_edge_ddi_net'], dtype=float)
if not (np.all(np.isfinite(true_edge_ddi)) and np.all(np.isfinite(true_edge_ddi_net)) and np.all(np.isfinite(max_false_edge_ddi)) and np.all(np.isfinite(max_false_edge_ddi_net))):
    raise RuntimeError('The DDI calculation is incomplete; run Part 2 until every omega is finished.')
true_edge_mean = true_edge_ddi
true_edge_std = np.std(true_edge_ddi_net, axis=1, ddof=1)
max_false_mean = max_false_edge_ddi
max_false_std = np.std(max_false_edge_ddi_net, axis=1, ddof=1)
(rho, p_value) = spearmanr(omega_list, true_edge_mean)
true_color = '#0072B2'
false_color = '#D55E00'
fit = np.polyfit(omega_list, true_edge_mean, 1)
fit_x = np.linspace(omega_list.min(), omega_list.max(), 100)
fit_values = np.polyval(fit, fit_x)
(fig, ax) = plt.subplots(figsize=(7.2, 7.2), facecolor='white')
fig.subplots_adjust(left=0.15, right=0.97, bottom=0.14, top=0.95)
ax.set_box_aspect(1)
ax.plot(omega_list, true_edge_mean, color=true_color, marker='o', markersize=7, linewidth=2.2, label='True edge')
ax.fill_between(omega_list, np.maximum(true_edge_mean - true_edge_std, 0), true_edge_mean + true_edge_std, color=true_color, alpha=0.16, linewidth=0)
ax.plot(omega_list, max_false_mean, color=false_color, marker='s', markersize=6, linewidth=2.0, linestyle='--', label='Maximum false edge')
ax.fill_between(omega_list, np.maximum(max_false_mean - max_false_std, 0), max_false_mean + max_false_std, color=false_color, alpha=0.13, linewidth=0)
ax.plot(fit_x, fit_values, color='#333333', linewidth=1.8, linestyle='-.', label='Linear fit', zorder=2.5)
ax.set_xlabel('Edge weight $\\omega$')
ax.set_ylabel('DDI')
ax.set_xticks(omega_list)
ax.ticklabel_format(axis='x', style='sci', scilimits=(-1, -1), useMathText=True)
ax.xaxis.get_offset_text().set_fontsize(10)
ax.set_xlim(omega_list[0] - 0.01, omega_list[-1] + 0.01)
data_top = np.max(np.concatenate((true_edge_mean + true_edge_std, max_false_mean + max_false_std)))
data_bottom = min(0.0, float(np.min(fit_values)))
lower_limit = data_bottom - 0.04 * (data_top - data_bottom) if data_bottom < 0 else 0.0
ax.set_ylim(lower_limit, 1.08 * data_top)
ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useMathText=True)
ax.yaxis.get_offset_text().set_fontsize(10)
ax.tick_params(axis='both', labelsize=12, width=1.0, direction='out')
ax.set_axisbelow(True)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(1.0)
ax.legend(loc='upper left', bbox_to_anchor=(0.02, 0.98), frameon=False, fontsize=12, handlelength=2.6, labelspacing=0.7)
rho_annotation = ax.text(0.96, 0.96, f"Spearman's $\\rho_s={rho:.2f}$", transform=ax.transAxes, ha='right', va='top', fontsize=12)
fig.savefig(FIGURE_PATH, dpi=600, facecolor='white')
plt.show()
print(FIGURE_PATH)
